<a href="https://colab.research.google.com/github/YefridC09/ST-554-Project1-Template/blob/main/Task3/Task_3_Project1_ST_554.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Task 3 - Project 1**

## Author: Yefrid Cordoba

## **Introduction**

The air quality database from the *UC Irvine machine learning repository* has the responses of a gas multisensor device, sensing the gas concentrations in an italian city hourly during 1 year, from March 2004 to February 2005.
Among the gases measured, benzene is one of the most important indicators of air quality, as this compound is currently linked to cancer development with long-term exposure.

There will be evaluated two predictive models. The mean squared error (`MSE`) will be used as the metric to determine which is best for predicting benzene concentration from carbon monoxide concentration, temperature, relative humidity, and absolute humidity. A simple linear regression model (`SLR`) with carbon monoxide concentration as the only predictor, and a multiple linear regression model (`MLR`) with all four predictors will be evaluated.

## **Importing data and modules**

First, using the Python'pakcages manager `pip`, install `ucimilrepo` in the Python environment, where the air quality data is stored.

In [2]:
!pip install ucimlrepo

From the `ucimilrepo`, we extract the list of air quality for our analysis. Then the data is filtered for all the non existing values wich are marked with a value of `-200` for each of the variables that are going to be evaluated:
- Benzene concentration (`C6H6(GT)`)
- Carbon monoxide concentration (`CO(GT)`)
- Temperature (`T`)
- Relative humidity (`RH`)
- Absolute humidity (`AH`)

In [8]:
import ucimlrepo as uci
import numpy as np
import pandas as pd
import sklearn as sk


air_quality = uci.fetch_ucirepo(id=360)
air_quality = air_quality.data.features
air_quality = air_quality[["Date", "Time", "C6H6(GT)",
                           "CO(GT)", "T", "RH", "AH"]]
#print(air_quality.info())
#Filtering the data for missing values
air_quality = air_quality.loc[
    (air_quality["C6H6(GT)"] != -200) &
    (air_quality["CO(GT)"] != -200) &
    (air_quality["T"] != -200) &
    (air_quality["RH"] != -200) &
    (air_quality["AH"] != -200)
]

air_quality.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7344 entries, 0 to 9356
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Date      7344 non-null   object 
 1   Time      7344 non-null   object 
 2   C6H6(GT)  7344 non-null   float64
 3   CO(GT)    7344 non-null   float64
 4   T         7344 non-null   float64
 5   RH        7344 non-null   float64
 6   AH        7344 non-null   float64
dtypes: float64(5), object(2)
memory usage: 459.0+ KB


As the objective of the analysis is to calculate the `MSE` for each model and compare the `SLR` and `MLR`, it is necessary to calculate the daily average for each variable of interest. However, the date is stored as an `object`, and the order will not be appropriate for further calculations.

First, we change the column date to a `pd.datetime` type object to ensure the dates are sorted properly.\
Then it is grouped by day (Column `"Date"`).\
The average for each variable is calculated per day.

In [9]:
#Chage the data type from the date column to pd.datetime
air_quality["Date"] = pd.to_datetime(air_quality["Date"])
air_quality.info()
#Calculate the mean value for each day across all the variables
air_quality = air_quality.groupby("Date")[["C6H6(GT)", "CO(GT)",
                                           "T", "RH", "AH"]].mean()
air_quality.head()

<class 'pandas.core.frame.DataFrame'>
Index: 7344 entries, 0 to 9356
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   Date      7344 non-null   datetime64[ns]
 1   Time      7344 non-null   object        
 2   C6H6(GT)  7344 non-null   float64       
 3   CO(GT)    7344 non-null   float64       
 4   T         7344 non-null   float64       
 5   RH        7344 non-null   float64       
 6   AH        7344 non-null   float64       
dtypes: datetime64[ns](1), float64(5), object(1)
memory usage: 459.0+ KB


,C6H6(GT),CO(GT),T,RH,AH
Date,,,,,
2004-03-10,8.450000,1.966667,12.033333,54.900000,0.765633
2004-03-11,8.269565,2.239130,9.826087,64.230435,0.777039
2004-03-12,12.177273,2.804545,11.618182,50.190909,0.665164
2004-03-13,11.121739,2.695652,13.121739,50.682609,0.733013
2004-03-14,9.830435,2.469565,16.182609,48.317391,0.849209


It is added a helper column to give index for each day that will be needed for the calculation of the sequential `MSE`.

In [10]:
air_quality["Day"] = range(1, len(air_quality) + 1)
print(air_quality.tail())
print(air_quality.info())


            C6H6(GT)    CO(GT)          T         RH        AH  Day
Date                                                               
2005-03-31  5.220833  1.387500  17.550000  50.083333  0.951917  343
2005-04-01  3.526087  1.108696  16.026087  35.404348  0.631135  344
2005-04-02  2.529167  0.854167  15.483333  32.225000  0.546167  345
2005-04-03  4.316667  1.141667  18.383333  33.695833  0.617583  346
2005-04-04  8.985714  2.078571  17.328571  41.842857  0.721250  347
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 347 entries, 2004-03-10 to 2005-04-04
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   C6H6(GT)  347 non-null    float64
 1   CO(GT)    347 non-null    float64
 2   T         347 non-null    float64
 3   RH        347 non-null    float64
 4   AH        347 non-null    float64
 5   Day       347 non-null    int64  
dtypes: float64(5), int64(1)
memory usage: 19.0 KB
None


It is confirmed that the final number of days for which we have data is 347 days.

## **Creating helper function to calculate the MSE per step**

As the data is collected over time, it is necessary to perform cross-validation sequentially, where the user specifies the day from which the function will predict and calculates the `MSE` until the last day the data was collected.

To perform this, it is necessary to define a function that takes a dataframe containing the predictors, a series containing the response variable, and the day until which the model will be trained. This regression will be used to predict the next day, and the mean squared error (MSE) will be calculated based on the predicted and measured values.

In [ ]:
import warnings

warnings.filterwarnings("ignore")  # hides all warnings from here on

# any code below will not show warnings


In [ ]:
def MSE2(X , Y: pd.Series, Day: int) -> float:
    """
    This function calculates the mean squared error for a given day.
    Uses just the next day to calculate the MSE.
    X: is a dataframe with the predictors
    Y: is a series with the target variable
    Day: is the day until the mean squared error is calculated
    (including this day).
    """
    #Slicing the predictors until the specified date to work as training set
    X_train = X.iloc[:Day]

    #Slicing the response variable until the specified date for training
    Y_train = Y.iloc[:Day]

    #Getting the predictor values to test the model
    X_test = X.iloc[[Day]]

    #Getting the response value with which we are going to compared to the predicted value
    Y_test = [Y.iloc[Day]]

    #Fit a linear regression to the training values
    reg = sk.linear_model.LinearRegression()
    reg.fit(X_train.values, Y_train.values)

    #Calculate the estimated point for the MSE calculation
    Y_pred = reg.predict(X_test)

    #MSE calculation
    MSE = sk.metrics.mean_squared_error(Y_test, Y_pred)
    return MSE

Next we test the function with two predictors and obtain the MSE

In [ ]:
MSE2(air_quality[["CO(GT)", "T"]], air_quality["C6H6(GT)"],250)

0.7161008160779453

In [ ]:
#Testing value for the MSE2 function
X = air_quality[["CO(GT)", "T"]].iloc[250]
print(X)
print(air_quality["C6H6(GT)"].iloc[250])
Y = -3.6565302977532053 + 5.14640409 * X['CO(GT)'] + 0.17034599 * X['T']
print(Y)
((air_quality["C6H6(GT)"].iloc[250] - Y)**2)

CO(GT)    2.221739
T         6.682609
Name: 2004-12-22 00:00:00, dtype: float64
8.069565217391304
8.915792644072884


np.float64(0.7161008576681284)

As seen in the user-defined function and the manual calculation of the `MSE`, both values match, confirming that the function accurately calculates the value to be used in the next step.

## **Construction of the function to calculate the cross-validation error**

To calculate the cross-validation, for each of the models being evaluated, it is required to iterate over the entire data set, starting from a specified day and continuing until the last day, and then calculate the average `MSE` for the `SLR` and `MLR` models using the required predictors for each model.

The function iterates over each day from the specified day to the last day, fitting a model using the user-specified predictors, predicting the next day, calculating the `MSE` against the actual value, summing all the `MSE`, and returning the average `MSE`.

In [ ]:
def CV_error(X, Y, Day):
    M_total = 0
    for i in range(Day, len(Y)):
        #print(M_total)
        #print(MSE(X, Y, i))
        M_total+= MSE2(X, Y, i)
        #print(M_total)
    return M_total/(len(Y)-Day)

## **Model fitting and cross-validation error**

As the objective of this work is to pick the best model to predict the data, it is necessary to evaluate a simple linear regression model (`SLR`) against a multiple linear regression model (`MLR`) and calculate the cross-validation error (`CV`), which is based on the MSE from each day, following a defined day for training the model.

### **Simple linear regression model**

In [11]:
#This code chunk will fit the regression model for CO concentration as the only predictor
reg1 = sk.linear_model.LinearRegression()
reg1.fit(air_quality[["CO(GT)"] ], air_quality["C6H6(GT)"])
print(reg1.intercept_, reg1.coef_)

0.644770948344199 [4.56536116]


The `SLR` model from the entire data set, where carbon monoxide concentration is the only predictor, is:

$\widehat{Benzene}_{[\mu g/m^3]} = 0.645 + 4.565 * CO_{[mg/m^3]}$

From which the `CV` error is calculated from 250 day until the last day of the dataset and the MSE is calculated and averaged across the validation days.

In [ ]:
CV_error(air_quality[["CO(GT)"]], air_quality["C6H6(GT)"],250)

7.402897202768712

### **Multiple linear regression model**

In [ ]:
#This code chunk will fit the multiple linear regression model
reg1 = sk.linear_model.LinearRegression()
reg1.fit(air_quality[["CO(GT)", "T", "RH", "AH"] ], air_quality["C6H6(GT)"]) #Fit a linear regression to the training values
print(reg1.intercept_, reg1.coef_)

-1.8377694729981364 [ 4.77080433  0.11973259 -0.01620259  0.68866811]


The `MLR` model for the CO concentration, temperature, relative humidity, and absolute humidity is:

$\widehat{Benzene}_{[\mu g/m^3]} = -1.838 + 4.771 * CO_{[mg/m^3]} + 0.120 * T_{°C} - 0.016 * RH_{\%} + 0.689 *AH$

With and `CV` error calculated from 250 day until the last day of the dataset.

In [ ]:
CV_error(air_quality[["CO(GT)", "T", "RH", "AH"]], air_quality["C6H6(GT)"],250)

5.0963115645839805

Based on cross-validation, the multiple linear regression (MLR) model shows lower MSE when predicting benzene concentration in the air from carbon monoxide concentration, temperature, relative humidity, and absolute humidity.

### **Discussion**

The `CV` error is an estimation of the mean-squared error (MSE) for each of the models. It provides a metric of how well each model predicts data not included in its training, even when two consecutive days have a relationship in the concentration of benzene in the air.\
The lower the `CV` for the model, the better the model is at predicting the benzene concentration. We conclude that the best model for predicting benzene concentration is the multiple linear regression model (`MLR`), with carbon monoxide concentration, ambient temperature, relative humidity, and absolute humidity as predictors.

$\widehat{Benzene}_{[\mu g/m^3]} = -1.838 + 4.771 * CO_{[mg/m^3]} + 0.120 * T_{°C} - 0.016 * RH_{\%} + 0.689 *AH$